# SmartHTR + SmartEval

Run this notebook in one Google Colab runtime. SmartHTR performs full-document OCR and SmartEval consumes the resulting text through a direct Python provider. No HTTP server, public URL, or tunnel is used.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

try:
    import torch
    print(f'Python: {sys.version.split()[0]}')
    print(f'Torch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('Torch is not installed in this runtime.')

## Project and dependency checks

Set `PROJECT_ROOT` before running this cell when the checkout lives somewhere other than `/content/ai-grading-platform`. The path must be visible to the Colab runtime.

In [ ]:
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/ai-grading-platform')).expanduser().resolve()
SMARTHTR_ROOT = PROJECT_ROOT / 'SmartHTR'
SMARTEVAL_ROOT = PROJECT_ROOT / 'SmartEval'
assert SMARTHTR_ROOT.is_dir(), f'SmartHTR is missing at {SMARTHTR_ROOT}'
assert SMARTEVAL_ROOT.is_dir(), f'SmartEval is missing at {SMARTEVAL_ROOT}'
for source_root in (SMARTHTR_ROOT, SMARTEVAL_ROOT):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))
os.environ.setdefault('SMARTHR_PROVIDER', 'local')

required_packages = ('cv2', 'fitz', 'numpy', 'sentence_transformers', 'torch', 'transformers')
missing = [package for package in required_packages if importlib.util.find_spec(package) is None]
if missing:
    raise RuntimeError(f'Missing runtime dependencies: {missing}')
print(f'Project root: {PROJECT_ROOT}')
print('SmartHTR and SmartEval will run in this same Python runtime.')

## Configure documents

Set `SMARTEVAL_INPUT_DIR` or assign the three paths below to documents available in the runtime. Question paper and model answer inputs use SmartEval's existing loaders; the student sheet is sent only to the existing full-page SmartHTR pipeline.

In [ ]:
INPUT_ROOT = Path(os.environ.get('SMARTEVAL_INPUT_DIR', SMARTEVAL_ROOT / 'inputs')).expanduser().resolve()
QUESTION_PAPER = INPUT_ROOT / 'question_paper.pdf'
MODEL_ANSWER = INPUT_ROOT / 'model_answer.pdf'
STUDENT_SHEET = INPUT_ROOT / 'student_answer_sheet.pdf'

for label, path in {'question paper': QUESTION_PAPER, 'model answer': MODEL_ANSWER, 'student sheet': STUDENT_SHEET}.items():
    assert path.is_file(), f'Missing {label}: {path}'
print('All input files are available.')

## Build the shared pipeline

`SharedRuntimeSmartHTRClient` constructs the existing SmartHTR pipeline only once. `EvaluationService` receives that same provider, so the evaluation run performs OCR once and then continues through parsing, mapping, semantic evaluation, marks, feedback, and report export.

In [ ]:
from config.settings import Settings as SmartEvalSettings
from integration.evaluation_service import EvaluationRequest, EvaluationService
from integration.smarthtr_client import SharedRuntimeSmartHTRClient
from integration.student_text_provider import SharedRuntimeSmartHTRStudentTextProvider, SmartHTRIntegrationService

settings = SmartEvalSettings.from_environment()
shared_client = SharedRuntimeSmartHTRClient(project_root=PROJECT_ROOT)
student_provider = SharedRuntimeSmartHTRStudentTextProvider(SmartHTRIntegrationService(shared_client))
evaluation_service = EvaluationService(settings, student_text_provider=student_provider)
print('Shared SmartHTR provider is ready. The existing OCR model loads lazily on the first sheet.')

## OCR, mapping, evaluation, marks, feedback, and export

In [ ]:
request = EvaluationRequest(
    question_paper_path=QUESTION_PAPER,
    model_answer_path=MODEL_ANSWER,
    student_answer_sheet_path=STUDENT_SHEET,
)
execution = evaluation_service.evaluate_and_export(request)
report = execution.report
ocr = execution.ocr_result
assert ocr is not None, 'OCR did not return a result.'
print(f'OCR status: {ocr.status}')
print(f'Pages: {len(ocr.pages)}')
print(f'Report status: {report.status}')

In [ ]:
ocr_view = {
    'status': ocr.status,
    'pages': [{'page_number': page.page_number, 'text': page.text} for page in ocr.pages],
    'full_text': ocr.full_text,
    'metadata': ocr.metadata,
    'warnings': ocr.warnings,
}
ocr_view

In [ ]:
quality_checks = {
    'ocr_has_text': bool(ocr.full_text.strip()),
    'ocr_page_count': len(ocr.pages),
    'ocr_warnings': list(ocr.warnings),
    'mapped_questions': report.summary.total_questions,
    'evaluated_questions': report.summary.evaluated_questions,
    'unanswered_questions': report.summary.unanswered_questions,
}
quality_checks

In [ ]:
for result in report.question_results:
    print(f'{result.question.question_id}: {result.marks.marks_awarded}/{result.marks.max_marks} - {result.feedback.overall_feedback}')
print(f'Total: {report.summary.awarded_marks}/{report.summary.evaluated_maximum_marks}')
print(f'JSON report: {execution.json_report_path}')
print(f'Text report: {execution.text_report_path}')